<a href="https://colab.research.google.com/github/Yousef-Shihade/information-retrieval-course-tasks/blob/main/Task_01_Porter_Stemmer/Porter_Stemmer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Task1 - porter stemmer
# Yousef Shihade

import re
from typing import List

# checking for consonant/vowel and also to measure m
VOWELS = set ( " aeiou " )

def is_consonant(w: str, i: int) -> bool:
    ch = w[i]
    if ch in VOWELS:
        return False
    if ch == 'y':         # Porter’s rule
        return i == 0 or not is_consonant(w, i - 1)
    return True

def measure_m(stem: str) -> int:
    if not stem:
        return 0
    m = 0
    i = 0
    L = len(stem)

    # we skip initial consonants
    while i < L and is_consonant(stem, i):
        i += 1
    while i < L:
        while i < L and not is_consonant(stem, i):
            i += 1
        if i < L:
            m += 1
        while i < L and is_consonant(stem, i):
            i += 1
    return m

# plural/initial suffix rules
def phase1(word: str) -> str:
    w = word
    # longest-suffix first
    if w.endswith("sses"):
        return w[:-2]  # sses becomes ss
    if w.endswith("ies"):
        return w[:-2]  # ies becomes i
    if w.endswith("ss"):
        return w       # ss stays the same
    if w.endswith("s"):
        return w[:-1]  # remove the last s
    return w

# explicit suffix replacement rules shown in slides
porter_sample_rules = [
    ("ational", "ate"),
    ("tional",  "tion"),
]

def phase2(word: str) -> str:
    w = word
    for suf, rep in sorted(porter_sample_rules, key=lambda x: -len(x[0])):  # the longest first
        if w.endswith(suf):
            stem = w[:-len(suf)]
            return stem + rep
    return w

# weight-sensitive rule (m > 1) for any word ending with the suffix (ement)
def phase3(word: str) -> str:
    w = word
    if w.endswith("ement"):
        stem = w[:-len("ement")]
        if measure_m(stem) > 1:  # weight-sensitive rule
            return stem
    return w

def porter(token: str) -> str:
    w = token.lower()
    if len(w) <= 2:
        return w
    w = phase1(w)
    w = phase2(w)
    w = phase3(w)
    return w

# Examples
examples = [
    "caresses", "ponies", "ties", "cats",
    "operational", "relational", "national", "institutional",
    "replacement", "cement","troubles","issues","dogs"]
print({x: porter(x) for x in examples})

{'caresses': 'caress', 'ponies': 'poni', 'ties': 'ti', 'cats': 'cat', 'operational': 'operate', 'relational': 'relate', 'national': 'nate', 'institutional': 'institution', 'replacement': 'replac', 'cement': 'cement', 'troubles': 'trouble', 'issues': 'issue', 'dogs': 'dog', 'eating': 'eating'}
